In [1]:
from pathlib import Path
import pandas as pd
import tarfile
import urllib.request

def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
    with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing = load_housing_data()

C:\Users\danmc\AppData\Local\Temp\ipykernel_11656\2839428726.py:13: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  housing_tarball.extractall(path="datasets")


In [2]:
housing.head()  # ← Typo here, should be "housing"

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [3]:
housing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


In [4]:
import numpy as np

def shuffle_and_split_data(data, test_ratio):
    shuffled_indices = np.random.permutation(len(data))
    test_set_size = int(len(data) * test_ratio)
    test_indices = shuffled_indices[:test_set_size]
    train_indices = shuffled_indices[test_set_size:]
    return data.iloc[train_indices], data.iloc[test_indices]

In [5]:
train_set, test_set = shuffle_and_split_data(housing, 0.2)
len(train_set)

16512

In [6]:
len(test_set)

4128

In [7]:
np.random.seed(42)

In [8]:
from zlib import crc32

def is_id_in_test_set(identifier, test_ratio):
    return crc32(np.int64(identifier)) < test_ratio * 2**32

def split_data_with_id_hash(data, test_ratio, id_column):
    ids = data[id_column]
    in_test_set = ids.apply(lambda id_: is_id_in_test_set(id_, test_ratio))
    return data.loc[~in_test_set], data.loc[in_test_set]

In [9]:
housing_with_id = housing.reset_index()  # adds an `index` column
train_set, test_set = split_data_with_id_hash(housing_with_id, 0.2, "index")

In [10]:
housing_with_id["id"] = housing["longitude"] * 1000 + housing["latitude"]
train_set, test_set = split_data_with_id_hash(housing_with_id, 0.2, "id")

In [11]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)

In [12]:
test_set["total_bedrooms"].isnull().sum()

np.int64(44)

In [13]:
# extra code – shows how to compute the 10.7% proba of getting a bad sample

from scipy.stats import binom

sample_size = 1000
ratio_female = 0.511
proba_too_small = binom(sample_size, ratio_female).cdf(485 - 1)
proba_too_large = 1 - binom(sample_size, ratio_female).cdf(535)
print(proba_too_small + proba_too_large)

0.10736798530929942


In [14]:
housing["income_cat"] = pd.cut(housing["median_income"],
                               bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                               labels=[1, 2, 3, 4, 5])

In [15]:
strat_train_set, strat_test_set = train_test_split(
    housing, test_size=0.2, stratify=housing["income_cat"], random_state=42)

In [16]:
strat_test_set["income_cat"].value_counts() / len(strat_test_set)

income_cat
3    0.350533
2    0.318798
4    0.176357
5    0.114341
1    0.039971
Name: count, dtype: float64

In [17]:
for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

In [18]:
housing = strat_train_set.copy()

In [19]:
corr_matrix = housing.corr(numeric_only=True)

In [20]:
corr_matrix["median_house_value"].sort_values(ascending=False)

median_house_value    1.000000
median_income         0.688380
total_rooms           0.137455
housing_median_age    0.102175
households            0.071426
total_bedrooms        0.054635
population           -0.020153
longitude            -0.050859
latitude             -0.139584
Name: median_house_value, dtype: float64

In [21]:
housing["rooms_per_house"] = housing["total_rooms"] / housing["households"]
housing["bedrooms_ratio"] = housing["total_bedrooms"] / housing["total_rooms"]
housing["people_per_house"] = housing["population"] / housing["households"]

In [22]:
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

In [23]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

In [24]:
housing_num = housing.select_dtypes(include=[np.number])
imputer.fit(housing_num)

SimpleImputer(strategy='median')

In [25]:
imputer.statistics_

array([-118.51  ,   34.26  ,   29.    , 2125.    ,  434.    , 1167.    ,
        408.    ,    3.5385])

In [26]:
X = imputer.transform(housing_num)

In [27]:
housing_tr = pd.DataFrame(X, columns=housing_num.columns,
                          index=housing_num.index)

In [28]:
housing_cat = housing[["ocean_proximity"]]
housing_cat.head(8)

,ocean_proximity
13096,NEAR BAY
14973,<1H OCEAN
3785,INLAND
14689,INLAND
20507,NEAR OCEAN
1286,INLAND
18078,<1H OCEAN
4396,NEAR BAY


In [29]:
from sklearn.preprocessing import OrdinalEncoder

ordinal_encoder = OrdinalEncoder()
housing_cat_encoded = ordinal_encoder.fit_transform(housing_cat)

In [30]:
from sklearn.preprocessing import OneHotEncoder

cat_encoder = OneHotEncoder()
housing_cat_1hot = cat_encoder.fit_transform(housing_cat)

In [31]:
from sklearn.preprocessing import StandardScaler

std_scaler = StandardScaler()
housing_num_std_scaled = std_scaler.fit_transform(housing_num)

In [32]:
from sklearn.pipeline import Pipeline

num_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("standardize", StandardScaler()),
])

In [33]:
from sklearn.pipeline import make_pipeline

num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())

In [34]:
from sklearn.preprocessing import FunctionTransformer

log_transformer = FunctionTransformer(np.log, inverse_func=np.exp)
log_pop = log_transformer.transform(housing[["population"]])

In [35]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_array, check_is_fitted

class StandardScalerClone(BaseEstimator, TransformerMixin):
    def __init__(self, with_mean=True):  # no *args or **kwargs!
        self.with_mean = with_mean

    def fit(self, X, y=None):  # y is required even though we don't use it
        X = check_array(X)  # checks that X is an array with finite float values
        self.mean_ = X.mean(axis=0)
        self.scale_ = X.std(axis=0)
        self.n_features_in_ = X.shape[1]  # every estimator stores this in fit()
        return self  # always return self!

    def transform(self, X):
        check_is_fitted(self)  # looks for learned attributes (with trailing _)
        X = check_array(X)
        assert self.n_features_in_ == X.shape[1]
        if self.with_mean:
            X = X - self.mean_
        return X / self.scale_

In [36]:
from sklearn.cluster import KMeans

class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, n_init=10,
                              random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self  # always return self!

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)
    
    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

In [37]:
from sklearn.compose import ColumnTransformer

num_attribs = ["longitude", "latitude", "housing_median_age", "total_rooms",
               "total_bedrooms", "population", "households", "median_income"]
cat_attribs = ["ocean_proximity"]

cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"))

preprocessing = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", cat_pipeline, cat_attribs),
])

In [38]:
from sklearn.metrics.pairwise import rbf_kernel

age_simil_35 = rbf_kernel(housing[["housing_median_age"]], [[35]], gamma=0.1)

In [39]:
import os
os.environ['LOKY_MAX_CPU_COUNT'] = '8'  # Set to your actual number of CPU cores
cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1., random_state=42)
similarities = cluster_simil.fit_transform(housing[["latitude", "longitude"]],
                                           sample_weight=housing_labels)

In [40]:
from sklearn.compose import make_column_selector, make_column_transformer


In [41]:
def column_ratio(X):
    return X[:, [0]] / X[:, [1]]

def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]  # feature names out

def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler())

log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, feature_names_out="one-to-one"),
    StandardScaler())
cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1., random_state=42)
default_num_pipeline = make_pipeline(SimpleImputer(strategy="median"),
                                     StandardScaler())
preprocessing = ColumnTransformer([
        ("bedrooms", ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
        ("rooms_per_house", ratio_pipeline(), ["total_rooms", "households"]),
        ("people_per_house", ratio_pipeline(), ["population", "households"]),
        ("log", log_pipeline, ["total_bedrooms", "total_rooms", "population",
                               "households", "median_income"]),
        ("geo", cluster_simil, ["latitude", "longitude"]),
        ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
    ],
    remainder=default_num_pipeline)  # one column remaining: housing_median_age

In [42]:
housing_prepared = preprocessing.fit_transform(housing)
housing_prepared.shape

(16512, 24)

In [43]:
preprocessing.get_feature_names_out()

array(['bedrooms__ratio', 'rooms_per_house__ratio',
       'people_per_house__ratio', 'log__total_bedrooms',
       'log__total_rooms', 'log__population', 'log__households',
       'log__median_income', 'geo__Cluster 0 similarity',
       'geo__Cluster 1 similarity', 'geo__Cluster 2 similarity',
       'geo__Cluster 3 similarity', 'geo__Cluster 4 similarity',
       'geo__Cluster 5 similarity', 'geo__Cluster 6 similarity',
       'geo__Cluster 7 similarity', 'geo__Cluster 8 similarity',
       'geo__Cluster 9 similarity', 'cat__ocean_proximity_<1H OCEAN',
       'cat__ocean_proximity_INLAND', 'cat__ocean_proximity_ISLAND',
       'cat__ocean_proximity_NEAR BAY', 'cat__ocean_proximity_NEAR OCEAN',
       'remainder__housing_median_age'], dtype=object)

In [44]:
from sklearn.linear_model import LinearRegression

lin_reg = make_pipeline(preprocessing, LinearRegression())
lin_reg.fit(housing, housing_labels)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder=Pipeline(steps=[('simpleimputer',
                                                              SimpleImputer(strategy='median')),
                                                             ('standardscaler',
                                                              StandardScaler())]),
                                   transformers=[('bedrooms',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('functiontransformer',
                                                                   FunctionTransformer(feature_names_out=<function ratio_name at 0x000...
                                                   'median_income']),
                                                 ('geo',
                                                  ClusterSimilarity(random_state=42),
                                                  ['latitude', 'longitude']),
                                                 ('cat',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x0000028744DFDFD0>)])),
                ('linearregression', LinearRegression())])

In [45]:
housing_predictions = lin_reg.predict(housing)
housing_predictions[:5].round(-2)  # -2 = rounded to the nearest hundred

array([242800., 375900., 127500.,  99400., 324600.])

In [46]:
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error

    def root_mean_squared_error(labels, predictions):
        return mean_squared_error(labels, predictions, squared=False)

lin_rmse = root_mean_squared_error(housing_labels, housing_predictions)
lin_rmse

68647.9568670666

# Support Vector Regression (SVR)

Now let's try Support Vector Regression with different kernels to see if we can improve upon the Linear Regression baseline.

In [47]:
from sklearn.svm import SVR
from sklearn.model_selection import cross_val_score

# Let's start with a basic SVR with RBF kernel using the full dataset
svr_reg = make_pipeline(preprocessing, SVR(kernel='rbf', C=100, gamma='scale', epsilon=0.1))
svr_reg.fit(housing, housing_labels)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder=Pipeline(steps=[('simpleimputer',
                                                              SimpleImputer(strategy='median')),
                                                             ('standardscaler',
                                                              StandardScaler())]),
                                   transformers=[('bedrooms',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('functiontransformer',
                                                                   FunctionTransformer(feature_names_out=<function ratio_name at 0x000...
                                                   'total_rooms', 'population',
                                                   'households',
                                                   'median_income']),
                                                 ('geo',
                                                  ClusterSimilarity(random_state=42),
                                                  ['latitude', 'longitude']),
                                                 ('cat',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x0000028744DFDFD0>)])),
                ('svr', SVR(C=100))])

In [48]:
# Using full dataset since SVR training time is acceptable (just a few minutes)
print(f"Using full housing dataset: {len(housing)} instances")

Using full housing dataset: 16512 instances


In [49]:
# Make predictions with SVR on the full dataset
svr_predictions = svr_reg.predict(housing)
svr_rmse = root_mean_squared_error(housing_labels, svr_predictions)
print(f"SVR RMSE: {svr_rmse:,.0f}")
print(f"Linear Regression RMSE: {lin_rmse:,.0f}")
print(f"Improvement: {((lin_rmse - svr_rmse) / lin_rmse * 100):,.1f}%")

SVR RMSE: 93,086
Linear Regression RMSE: 68,648
Improvement: -35.6%


In [50]:
# Let's compare different SVR kernels
from sklearn.model_selection import cross_val_score
import time

kernels = ['linear', 'rbf', 'poly']
svr_results = {}

for kernel in kernels:
    print(f"Testing SVR with {kernel} kernel...")
    start_time = time.time()
    
    if kernel == 'poly':
        svr = make_pipeline(preprocessing, SVR(kernel=kernel, degree=3, C=100, epsilon=0.1))
    else:
        svr = make_pipeline(preprocessing, SVR(kernel=kernel, C=100, epsilon=0.1))
    
    # Use 3-fold cross-validation on full dataset
    cv_scores = cross_val_score(svr, housing, housing_labels, 
                               scoring='neg_root_mean_squared_error', cv=3)
    
    training_time = time.time() - start_time
    mean_rmse = -cv_scores.mean()
    std_rmse = cv_scores.std()
    
    svr_results[kernel] = {
        'rmse': mean_rmse,
        'std': std_rmse,
        'time': training_time
    }
    
    print(f"{kernel.upper()} SVR - RMSE: {mean_rmse:,.0f} (±{std_rmse:,.0f}) - Time: {training_time:.1f}s")

print(f"\nFor comparison, Linear Regression RMSE: {lin_rmse:,.0f}")

Testing SVR with linear kernel...
LINEAR SVR - RMSE: 78,666 (±3,987) - Time: 10.2s
Testing SVR with rbf kernel...
RBF SVR - RMSE: 98,387 (±847) - Time: 17.3s
Testing SVR with poly kernel...
POLY SVR - RMSE: 216,572 (±149,645) - Time: 11.1s

For comparison, Linear Regression RMSE: 68,648


In [51]:
# Hyperparameter tuning for the best kernel
from sklearn.model_selection import GridSearchCV

# Based on the results above, let's tune the best performing kernel
# (You can modify this based on which kernel performed best)

print("Performing hyperparameter tuning for RBF kernel on full dataset...")

param_grid = {
    'svr__C': [10, 100, 1000],
    'svr__gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
    'svr__epsilon': [0.01, 0.1, 0.2]
}

svr_pipeline = make_pipeline(preprocessing, SVR(kernel='linear'))

# Using 3-fold CV on full dataset since training time is acceptable
grid_search = GridSearchCV(
    svr_pipeline, 
    param_grid, 
    cv=3, 
    scoring='neg_root_mean_squared_error',
    verbose=1,
    n_jobs=-1  # Use all available cores
)

grid_search.fit(housing, housing_labels)

Performing hyperparameter tuning for RBF kernel on full dataset...
Fitting 3 folds for each of 45 candidates, totalling 135 fits


GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('columntransformer',
                                        ColumnTransformer(remainder=Pipeline(steps=[('simpleimputer',
                                                                                     SimpleImputer(strategy='median')),
                                                                                    ('standardscaler',
                                                                                     StandardScaler())]),
                                                          transformers=[('bedrooms',
                                                                         Pipeline(steps=[('simpleimputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('functiontransformer',
                                                                                          FunctionTransformer(feature_names_ou...
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('onehotencoder',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         <sklearn.compose._column_transformer.make_column_selector object at 0x0000028744DFDFD0>)])),
                                       ('svr', SVR(kernel='linear'))]),
             n_jobs=-1,
             param_grid={'svr__C': [10, 100, 1000],
                         'svr__epsilon': [0.01, 0.1, 0.2],
                         'svr__gamma': ['scale', 'auto', 0.001, 0.01, 0.1]},
             scoring='neg_root_mean_squared_error', verbose=1)

In [52]:
# Results of hyperparameter tuning
print("Best parameters found:")
print(grid_search.best_params_)
print(f"\nBest cross-validation RMSE: {-grid_search.best_score_:,.0f}")

# Get the best model
best_svr = grid_search.best_estimator_

# Make predictions with the best model on the full dataset
best_svr_predictions = best_svr.predict(housing)
best_svr_rmse = root_mean_squared_error(housing_labels, best_svr_predictions)

print(f"\nFinal Model Performance:")
print(f"Best SVR RMSE: {best_svr_rmse:,.0f}")
print(f"Linear Regression RMSE: {lin_rmse:,.0f}")
print(f"Improvement over Linear Regression: {((lin_rmse - best_svr_rmse) / lin_rmse * 100):,.1f}%")

Best parameters found:
{'svr__C': 1000, 'svr__epsilon': 0.01, 'svr__gamma': 'scale'}

Best cross-validation RMSE: 71,718

Final Model Performance:
Best SVR RMSE: 71,204
Linear Regression RMSE: 68,648
Improvement over Linear Regression: -3.7%


In [53]:
# Final performance comparison
print("\n" + "="*50)
print("FINAL MODEL COMPARISON")
print("="*50)

print(f"SVR (best tuned model) RMSE: {best_svr_rmse:,.0f}")
print(f"Linear Regression RMSE: {lin_rmse:,.0f}")
print(f"SVR improvement over Linear Regression: {((lin_rmse - best_svr_rmse) / lin_rmse * 100):,.1f}%")

if best_svr_rmse < lin_rmse:
    print("✅ SVR outperforms Linear Regression")
else:
    print("❌ SVR does not improve over Linear Regression")


FINAL MODEL COMPARISON
SVR (best tuned model) RMSE: 71,204
Linear Regression RMSE: 68,648
SVR improvement over Linear Regression: -3.7%
❌ SVR does not improve over Linear Regression


In [54]:
(68648-73656)/68648

-0.07295187041137396

# RandomizedSearchCV

Now let's compare `RandomizedSearchCV` with `GridSearchCV`. RandomizedSearchCV is often more efficient as it:
- Samples random parameter combinations instead of trying all combinations
- Can explore a much larger parameter space in the same time
- Often finds equally good or better results than GridSearchCV
- Allows you to control the number of iterations (budget)

In [55]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, loguniform
import time

print("Performing RandomizedSearchCV for SVR...")

# Define a larger parameter space for RandomizedSearchCV
# Using distributions instead of discrete lists
param_distributions = {
    'svr__C': loguniform(1, 1000),          # Log-uniform distribution from 1 to 1000
    'svr__gamma': loguniform(0.0001, 1),    # Log-uniform from 0.0001 to 1
    'svr__epsilon': uniform(0.01, 0.5),     # Uniform from 0.01 to 0.51
}

# Create SVR pipeline (you can change kernel here)
svr_pipeline_random = make_pipeline(preprocessing, SVR(kernel='rbf'))

# RandomizedSearchCV with budget control
start_time = time.time()

random_search = RandomizedSearchCV(
    svr_pipeline_random,
    param_distributions=param_distributions,
    n_iter=50,  # Number of parameter settings to sample
    cv=3,
    scoring='neg_root_mean_squared_error',
    verbose=1,
    n_jobs=-1,
    random_state=42  # For reproducibility
)

random_search.fit(housing, housing_labels)
random_time = time.time() - start_time

print(f"\nRandomizedSearchCV completed in {random_time:.1f} seconds")

Performing RandomizedSearchCV for SVR...
Fitting 3 folds for each of 50 candidates, totalling 150 fits

RandomizedSearchCV completed in 194.8 seconds


In [56]:
# Compare RandomizedSearchCV vs GridSearchCV results
print("\n" + "="*60)
print("RANDOMIZEDSEARCHCV vs GRIDSEARCHCV COMPARISON")
print("="*60)

# RandomizedSearchCV results
print("RandomizedSearchCV Results:")
print(f"Best parameters: {random_search.best_params_}")
print(f"Best CV RMSE: {-random_search.best_score_:,.0f}")

# Get best RandomizedSearchCV model predictions
best_random_svr = random_search.best_estimator_
random_predictions = best_random_svr.predict(housing)
random_rmse = root_mean_squared_error(housing_labels, random_predictions)
print(f"Training RMSE: {random_rmse:,.0f}")

print(f"\nGridSearchCV Results (for comparison):")
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV RMSE: {-grid_search.best_score_:,.0f}")
print(f"Training RMSE: {best_svr_rmse:,.0f}")

# Performance comparison
print(f"\nPerformance Comparison:")
print(f"RandomizedSearchCV RMSE: {random_rmse:,.0f}")
print(f"GridSearchCV RMSE: {best_svr_rmse:,.0f}")
print(f"Linear Regression RMSE: {lin_rmse:,.0f}")

# Efficiency comparison
grid_time = "N/A"  # We don't have grid search timing from previous run
print(f"\nEfficiency Comparison:")
print(f"RandomizedSearchCV time: {random_time:.1f}s ({50} parameter combinations)")
print(f"GridSearchCV time: {grid_time} (would try {3*5*3} = 45 combinations)")

# Best overall model
if random_rmse < best_svr_rmse:
    print(f"\n🎯 RandomizedSearchCV found a better model!")
    best_overall_rmse = random_rmse
    best_method = "RandomizedSearchCV"
else:
    print(f"\n🎯 GridSearchCV performed better")
    best_overall_rmse = best_svr_rmse  
    best_method = "GridSearchCV"

print(f"Best model improvement over Linear Regression: {((lin_rmse - best_overall_rmse) / lin_rmse * 100):,.1f}%")


RANDOMIZEDSEARCHCV vs GRIDSEARCHCV COMPARISON
RandomizedSearchCV Results:
Best parameters: {'svr__C': np.float64(702.5166339242153), 'svr__epsilon': np.float64(0.4928160165372797), 'svr__gamma': np.float64(0.17123375973163968)}
Best CV RMSE: 78,246
Training RMSE: 73,656

GridSearchCV Results (for comparison):
Best parameters: {'svr__C': 1000, 'svr__epsilon': 0.01, 'svr__gamma': 'scale'}
Best CV RMSE: 71,718
Training RMSE: 71,204

Performance Comparison:
RandomizedSearchCV RMSE: 73,656
GridSearchCV RMSE: 71,204
Linear Regression RMSE: 68,648

Efficiency Comparison:
RandomizedSearchCV time: 194.8s (50 parameter combinations)
GridSearchCV time: N/A (would try 45 = 45 combinations)

🎯 GridSearchCV performed better
Best model improvement over Linear Regression: -3.7%


# SelectFromModel

In [57]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor
selector_pipeline = Pipeline([
    ('preprocessing', preprocessing),
    ('selector', SelectFromModel(RandomForestRegressor(random_state=42),
                                 threshold=0.005)),  # min feature importance
    ('svr', SVR(C=random_search.best_params_["svr__C"],
                gamma=random_search.best_params_["svr__gamma"],
                kernel="linear")),
])

In [58]:
svr_rnd_search_rmse = -random_search.best_score_
svr_rnd_search_rmse

np.float64(78245.83404967979)